# Hospital Appointment No-Show Prediction: Exploratory Data Analysis (EDA)
**Author:** Final Year B.Tech Computer Science (Data Science / ML)

**Assessment:** ALGIONS AI LABS PRIVATE LIMITED

### Objective
Analyze factors influencing outpatient appointment attendance and verify data hygiene, distributions, and temporal relationships before machine learning pipeline construction.

## 1. Load Dataset and Inspect Structure

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load appointments dataset
df = pd.read_csv('../data/appointments.csv')
print(f'Shape of dataset: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## 2. Data Cleaning & Integrity Checks
Verify missing values, duplicates, date consistency, and logical constraints.

In [ ]:
# Check missing values
print('--- Missing Values Check ---')
print(df.isnull().sum())

# Check duplicate records
print(f'\nTotal duplicate rows: {df.duplicated().sum()}')

# Verify logical constraints: previous_no_shows <= previous_appointments
invalid_history = (df['previous_no_shows'] > df['previous_appointments']).sum()
print(f'Invalid history records (no-shows > total appointments): {invalid_history}')

# Verify lead time >= 0
invalid_lead_times = (df['lead_time_days'] < 0).sum()
print(f'Negative lead time records: {invalid_lead_times}')

## 3. Summary Statistics

In [ ]:
df.describe()

## 4. Key Exploratory Visualizations & Insights

### A. Target Variable Distribution
*Explanation:* The target variable exhibits a 73.4% attendance vs. 26.6% no-show split. This realistic class imbalance confirms that standard accuracy alone is insufficient, and evaluation must prioritize Recall, Precision, and ROC-AUC.

In [ ]:
plt.figure(figsize=(6, 4))
counts = df['no_show'].value_counts()
sns.barplot(x=['Attended (0)', 'No-Show (1)'], y=counts.values, palette=['#2ecc71', '#e74c3c'], edgecolor='black')
plt.title('Target Distribution (Attended vs No-Show)')
plt.ylabel('Appointments')
plt.show()

### B. No-Show Rate Across Medical Departments
*Explanation:* Routine departments such as Dermatology and General Medicine display higher no-show rates (~29-30%), whereas specialized and critical departments like Cardiology show significantly lower no-show rates (~20%).

In [ ]:
plt.figure(figsize=(8, 4))
dept_rates = df.groupby('department')['no_show'].mean() * 100
dept_rates.sort_values(ascending=False).plot(kind='bar', color='#3498db', edgecolor='black')
plt.ylabel('No-Show Rate (%)')
plt.title('Department vs. No-Show Rate')
plt.xticks(rotation=25)
plt.show()

### C. No-Show Rate by Age Category
*Explanation:* Younger adults (18-35) have the highest no-show rate (~30%), likely driven by schedule conflicts and mobility. Senior patients (71+) show the lowest no-show rate (~21%), indicating higher adherence to scheduled visits.

In [ ]:
plt.figure(figsize=(7, 4))
age_rates = df.groupby('age_group')['no_show'].mean() * 100
age_rates.plot(kind='bar', color='#9b59b6', edgecolor='black')
plt.ylabel('No-Show Rate (%)')
plt.title('Age Group vs. No-Show Rate')
plt.xticks(rotation=20)
plt.show()

### D. Advance Lead Time vs. No-Show Rate
*Explanation:* Same-day bookings have very low no-show rates (<15%). As lead time expands beyond 14 to 30 days, no-show rates climb substantially (>35%), establishing lead time as a strong predictive signal.

In [ ]:
bins = [-1, 0, 3, 7, 14, 30, 60]
bin_labels = ['Same Day', '1-3d', '4-7d', '8-14d', '15-30d', '>30d']
df['lead_bin'] = pd.cut(df['lead_time_days'], bins=bins, labels=bin_labels)
df.groupby('lead_bin', observed=True)['no_show'].mean().mul(100).plot(kind='bar', color='#e67e22', edgecolor='black')
plt.ylabel('No-Show Rate (%)')
plt.title('Lead Time Days vs. No-Show Rate')
plt.show()

### E. Prior No-Show History vs. Current Attendance
*Explanation:* Patients with zero prior missed appointments have an ~18% no-show rate, while patients with 3 or more historical no-shows reach missed appointment rates over 50%. Past reliability is a dominant predictor.

In [ ]:
df['prev_group'] = df['previous_no_shows'].apply(lambda x: str(x) if x < 4 else '4+')
df.groupby('prev_group')['no_show'].mean().mul(100).plot(kind='bar', color='#c0392b', edgecolor='black')
plt.ylabel('Current No-Show Rate (%)')
plt.title('Prior No-Shows vs. Current No-Show Probability')
plt.show()

## 5. Key Takeaways for Machine Learning Modeling
1. **Leakage Safety:** Target variable and appointment IDs must be strictly excluded from features.
2. **Feature Set:** `lead_time_days`, `previous_appointments`, `previous_no_shows`, `age_group`, `appointment_type`, `department`, `booking_channel`, `sms_opt_in`.
3. **Preprocessing:** OneHotEncoding for categories, StandardScaler for numericals, wrapped in a scikit-learn `Pipeline`.
4. **Split Strategy:** Chronological split (70% train / 15% validation / 15% test) to prevent temporal data leakage.